# pq-verify v2.6.0 — One-Click Demo

Run all cells (Runtime -> Run all). ~3 minutes to 240/240 NIST ACVP vectors.

Upload `pq_verify_v2_6_0.py` when the file picker appears in Cell 2.

In [ ]:
# Cell 1: dependencies (~60s)
!apt-get install -y -qq coq gcc g++ > /dev/null 2>&1
!pip install kyber-py -q --break-system-packages
print('dependencies installed')

In [ ]:
# Cell 2: upload + load the stack (runs 160-test self-suite)
from google.colab import files
up = files.upload()
fn = list(up.keys())[0]
exec(open(fn).read())

In [ ]:
# Cell 3: full NIST ACVP — all 12 groups, 240/240
pqverify_acvp()

In [ ]:
# Cell 4: parameter security check
pqverify_params('ML-KEM-1024')

In [ ]:
# Cell 5: native full-KEM at Level 5
pqverify_kem(k=4)

In [ ]:
# Cell 6: audit a compiled .so (build a demo NTT, then audit it)
ntt_c = '''#include <stdint.h>
#define Q 3329
#define N 256
static uint16_t pw(uint16_t b,uint32_t e,uint16_t q){uint32_t r=1,x=b;while(e){if(e&1)r=r*x%q;x=x*x%q;e>>=1;}return(uint16_t)r;}
static uint32_t br(uint32_t x){uint32_t r=0;for(int i=0;i<7;i++){r=(r<<1)|(x&1);x>>=1;}return r;}
static uint16_t z[128];static int rdy=0;
void ntt(uint16_t f[N]){if(!rdy){for(int i=0;i<128;i++)z[i]=pw(17,br(i),Q);rdy=1;}int k=1;
for(int L=128;L>=2;L>>=1)for(int s=0;s<N;s+=2*L){uint16_t zz=z[k++];for(int j=s;j<s+L;j++){
uint16_t t=(uint16_t)(((uint32_t)zz*f[j+L])%Q);f[j+L]=(uint16_t)((f[j]+Q-t)%Q);f[j]=(uint16_t)((f[j]+t)%Q);}}}'''
open('kyber_ntt.c','w').write(ntt_c)
import os; os.system('gcc -O3 -shared -fPIC -o libkyber_ntt.so kyber_ntt.c')
ntt = pqverify_load_so('./libkyber_ntt.so','ntt')
pqverify_kat(ntt, k=4)
pqverify_leakage()